# Pelatihan Model YOLOv8-Nano: Prenava 'Isi Piringku'
**Tugas Akhir Individu 3 - Deep Learning & Computer Vision Specialist**  
Fakultas Rekayasa Industri — Telkom University

Notebook ini digunakan untuk melatih model deteksi objek 5 kelas makanan nusantara menggunakan akselerasi **GPU Tesla T4 (Google Colab)**.

In [ ]:
# 1. Periksa Akselerasi GPU Colab
!nvidia-smi

In [ ]:
# 2. Pasang Library yang Dibutuhkan
!pip install --upgrade pip ultralytics roboflow

In [ ]:
# 3. Unduh Dataset dari Roboflow (Ganti dengan API Key dan Project Tim Anda)
from roboflow import Roboflow

# Masukkan API key dari akun Roboflow Anda
rf = Roboflow(api_key="MASUKKAN_ROBOFLOW_API_KEY_ANDA")
project = rf.workspace("MASUKKAN_WORKSPACE_ANDA").project("prenava-isi-piringku")
version = project.version(1)
dataset = version.download("yolov8")

print(f"Dataset berhasil diunduh ke: {dataset.location}")

In [ ]:
# 4. Training Model YOLOv8-Nano (Transfer Learning)
from ultralytics import YOLO

# Muat base model pre-trained
model = YOLO("yolov8n.pt")

# Jalankan pelatihan 75 epoch pada GPU T4
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=75,
    imgsz=640,
    batch=16,
    name="prenava_food_yolov8n",
    device=0,  # GPU Tesla T4
    save=True,
    plots=True
)

In [ ]:
# 5. Evaluasi Metrik Akademik (mAP@0.5, PR Curve, Confusion Matrix)
metrics = model.val(
    data=f"{dataset.location}/data.yaml",
    imgsz=640,
    batch=16,
    device=0
)

print(f"mAP@0.5      : {metrics.box.map50 * 100:.2f}%")
print(f"mAP@0.5:0.95 : {metrics.box.map * 100:.2f}%")
print(f"Precision    : {metrics.box.mp * 100:.2f}%")
print(f"Recall       : {metrics.box.mr * 100:.2f}%")

In [ ]:
# 6. Download Bobot Terbaik (best.pt) ke Komputer Lokal
from google.colab import files
import shutil

best_weight_path = f"{results.save_dir}/weights/best.pt"
print(f"Mengunduh file bobot terbaik: {best_weight_path}")
files.download(best_weight_path)
# Simpan file yang terunduh ke: ml-service-food/models/best.pt di komputer Anda